# Лабораторная работа №3. Фонетическая транскрипция (G2P)

Курс «Синтез речи», Университет ИТМО, 2026.

**Цель:** преобразовать нормализованный русский текст в фонемы, получить временные границы фонем для корпуса RUSLAN и оценить качество результата.

## 1. Постановка задачи

1. Подготовить словарь произношений для всех слов принятого корпуса.
2. Обработать отсутствующие в словаре слова (OOV) G2P-моделью.
3. Сопоставить аудио и текст на уровне фонем и получить их длительности.
4. Зафиксировать фонемный алфавит для акустической модели лабораторной №4.
5. Рассчитать PER, сравнить с baseline и выполнить структурный аудит.

## 2. Входные данные и признаки

Используются нормализованные тексты лабораторной №1, 22 100 WAV RUSLAN и word-level разметка лабораторной №2. Для G2P входом служит последовательность Unicode-графем слова. Модель возвращает последовательность IPA/MFA-фонем с контекстными эффектами русского языка: мягкостью, редукциями, оглушением/озвончением и вариантами произношения.

## 3. Метод

Использован Montreal Forced Aligner 3.4.3 и официальный комплект `russian_mfa`: произносительный словарь, WFST G2P и HMM-GMM акустическая модель. Словарные слова получают одно или несколько эталонных произношений, OOV — предсказание G2P. Затем forced alignment ищет временные интервалы слов и фонем, согласованные одновременно с аудио и заданным текстом. Это предпочтительнее посимвольных правил, которые не моделируют контекст и не дают временных границ.

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'labs').is_dir())
DATA = ROOT / 'data' / 'lab3_g2p'
print('Корень проекта:', ROOT)
print('Каталог результатов:', DATA)

Корень проекта: C:\work projects\ITMO\tts_labs_itmo
Каталог результатов: C:\work projects\ITMO\tts_labs_itmo\data\lab3_g2p


## 4. Корпус и покрытие словаря

In [2]:
corpus_stats = json.loads((DATA / 'mfa_corpus/corpus_stats.json').read_text(encoding='utf-8'))
lexicon_stats = json.loads((DATA / 'lexicon_stats.json').read_text(encoding='utf-8'))
display(pd.DataFrame({
    'показатель': ['фразы', 'словоупотребления', 'уникальные слова', 'словарные слова', 'OOV', 'покрытие словаря'],
    'значение': [corpus_stats['utterances'], corpus_stats['words'], corpus_stats['unique_words'], lexicon_stats['known_words'], lexicon_stats['oov_words'], f"{lexicon_stats['type_coverage']:.2%}"],
}))
print('Словарь корпуса, записей с вариантами:', lexicon_stats['corpus_dictionary_entries'])
print('Приведение ɕ → ɕː:', lexicon_stats['remapped_phone_occurrences'])

,показатель,значение
0,фразы,22100
1,словоупотребления,254972
2,уникальные слова,48121
3,словарные слова,39090
4,OOV,9031
5,покрытие словаря,81.23%


Словарь корпуса, записей с вариантами: 58386
Приведение ɕ → ɕː: {'ɕ->ɕː': 17}


## 5. Оценка G2P

Phone Error Rate: $PER=(S+D+I)/N$, где $S$, $D$, $I$ — замены, удаления и вставки фонем, а $N$ — число эталонных фонем. Baseline независимо заменяет каждую букву одной фонемой и не знает контекста. Тест — фиксированная выборка 1 000 однозначных слов (seed 42) из официального словаря, пересекающихся со словарём RUSLAN.

In [3]:
g2p_metrics = json.loads((DATA / 'g2p_metrics.json').read_text(encoding='utf-8'))
comparison = pd.DataFrame([
    {'метод': 'MFA G2P', 'PER': g2p_metrics['model']['per'], 'точность слова': g2p_metrics['model']['word_accuracy']},
    {'метод': 'посимвольный baseline', 'PER': g2p_metrics['baseline']['per'], 'точность слова': g2p_metrics['baseline']['word_accuracy']},
])
display(comparison.style.format({'PER': '{:.3%}', 'точность слова': '{:.1%}'}))
errors = pd.read_csv(DATA / 'g2p_errors.csv')
display(errors.query('distance > 0').head(13))

,метод,PER,точность слова
0,MFA G2P,0.215%,98.7%
1,посимвольный baseline,48.601%,5.5%


,word,reference,prediction,distance
0,плита,p ʎ ɪ t̪ a,p ʎ i t̪ ə,2
1,тсс,s̪ː,tɕ s̪,2
2,ангел,a n̪ ɟ ɪ ɫ,ɐ n̪ ɟ ɪ ɫ,1
3,афиши,ɐ fʲ i ʂ ɨ,ɐ fʲ ɪ ʂ ɨ,1
4,без,bʲ ɪ s̪,bʲ e s̪,1
5,бесчисленных,bʲ ɪ ɕː tɕ i sʲ ʎ ɪ n̪ː ɨ x,bʲ ɪ ɕ tɕ i sʲ ʎ ɪ n̪ː ɨ x,1
6,джованни,dʐː o v ə ɲ ɪ,dʐː o v ə n̪ ɪ,1
7,дяде,dʲ æ dʲ e,dʲ æ dʲ ɪ,1
8,ер,j e r,j ɪ r,1
9,кулак,k ʊ ɫ a k,k ʊ ɫ ɐ k,1


Оценка показывает преимущество контекстной G2P-модели. Ошибки найдены в 13 из 1 000 слов: десять связаны с выбором гласного аллофона и редукции (`а/ɐ/ə`, `е/ɪ`, `и/ɪ`), остальные — с палатализацией в заимствовании, сложным стыком согласных и междометием `тсс`. В 11 случаях расстояние равно одной фонеме, в двух — двум. При этом оценка оптимистична: эталоны взяты из того же официального лексического ресурса, а не созданы независимым фонетистом.

## 6. Фонемный алфавит

Алфавит основан на IPA/MFA. Палатализация отмечается `ʲ`, долгота — `ː`; `sil` — тишина, `spn` — неизвестный речевой фрагмент.

In [4]:
inventory = pd.read_csv(DATA / 'phoneme_inventory.csv')
print('Символов в словаре:', len(inventory))
display(inventory.head(20))

Символов в словаре: 90


,phone,lexicon_count,is_special
0,a,17598,0
1,b,5568,0
2,bʲ,2279,0
3,bʲː,1,0
4,bː,8,0
5,c,4672,0
6,cː,14,0
7,dzʲː,6,0
8,dʐː,209,0
9,dʲ,4507,0


## 7. Результат фонемного выравнивания

In [5]:
alignment_stats = json.loads((DATA / 'alignment_stats.json').read_text(encoding='utf-8'))
keys = ['utterances', 'mfa_utterances', 'fallback_utterances', 'phone_intervals', 'speech_phone_intervals', 'phone_inventory_size', 'mean_phone_duration_ms']
display(pd.Series({k: alignment_stats[k] for k in keys}, name='значение').to_frame())
top_phones = pd.Series(alignment_stats['phone_counts']).sort_values(ascending=False).head(20)
display(top_phones.rename('число интервалов').to_frame())

,значение
utterances,2.210000e+04
mfa_utterances,2.209900e+04
fallback_utterances,1.000000e+00
phone_intervals,1.500906e+06
speech_phone_intervals,1.442428e+06
phone_inventory_size,8.900000e+01
mean_phone_duration_ms,7.519195e+01


,число интервалов
ɪ,118016
ə,109931
ɐ,76075
a,64667
t̪,63981
sil,58478
j,56181
n̪,51517
e,50313
r,46202


In [6]:
durations = pd.read_csv(DATA / 'phoneme_alignments.csv', usecols=['duration_s', 'is_silence'])
speech_ms = durations.loc[(durations['is_silence'] == 0) & (durations['duration_s'] <= 0.4), 'duration_s'] * 1000
display(speech_ms.describe(percentiles=[.5, .9, .95, .99]).to_frame('duration_ms'))

,duration_ms
count,1.442376e+06
mean,7.257202e+01
std,4.105503e+01
min,4.943000e+00
50%,7.000000e+01
90%,1.200000e+02
95%,1.400000e+02
99%,1.900000e+02
max,4.000000e+02


In [7]:
utterances = pd.read_csv(DATA / 'utterance_phonemes.csv')
example = utterances.query("utterance_id == '000000_RUSLAN'").iloc[0]
print('000000_RUSLAN — фонем:', example['phone_count'])
print(example['phones'])

000000_RUSLAN — фонем: 30
s̪ t̪ rʲ ɪ v o ʐ n̪ ɨ m tɕ u s̪ t̪ v ə m bʲ ɪ r u sʲ j a z̪ ə pʲ ɪ r o


## 8. Согласованность с лабораторной №2

Это не сравнение с ручным gold standard, а проверка согласия двух независимых автоматических методов. CTC-интервалы лабораторной №2 обычно уже, чем интервалы слов MFA.

In [8]:
boundary_metrics = json.loads((DATA / 'word_boundary_metrics.json').read_text(encoding='utf-8'))
display(pd.DataFrame({
    'граница': ['начало слова', 'конец слова'],
    'median_abs_error_ms': [boundary_metrics['start_absolute_error']['median_ms'], boundary_metrics['end_absolute_error']['median_ms']],
    'p95_abs_error_ms': [boundary_metrics['start_absolute_error']['p95_ms'], boundary_metrics['end_absolute_error']['p95_ms']],
}))
print('Сравнено слов:', boundary_metrics['compared_words'])
print('Несовпадений подписей:', boundary_metrics['word_label_mismatches'])

,граница,median_abs_error_ms,p95_abs_error_ms
0,начало слова,52.490,109.3949
1,конец слова,37.285,141.6183


Сравнено слов: 254972
Несовпадений подписей: 0


## 9. Автоматическая проверка полного результата

In [9]:
audit = subprocess.run([sys.executable, str(ROOT / 'labs/lab3_g2p/audit_lab3.py'), '--require-complete'], cwd=ROOT, text=True, capture_output=True, check=True)
print(audit.stdout)
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT / 'labs/lab3_g2p'), '-p', 'test_*.py'], cwd=ROOT, text=True, capture_output=True, check=True)
print('Автотесты:', tests.stderr.strip().splitlines()[-1])

Expected utterances: 22100
Parsed utterances: 22100
Phone intervals: 1500906
Missing utterances: 0
Unexpected utterances: 0
Duplicate utterance rows: 0
Bad timing groups: 0
Empty speech groups: 0
Missing TextGrids for parsed rows: 0



Автотесты: OK


## 10. Выводы и ограничения

- Подготовлены воспроизводимый G2P и единый IPA/MFA-алфавит.
- Контекстная модель значительно превосходит baseline: PER 0,215% против 48,601%.
- Получена полная фонемная разметка 22 100 фраз и длительности для FastSpeech 2.
- 22 099 фраз выровнены MFA. Запись `005321_RUSLAN` явно помечена как `lab2_uniform_fallback`; её интервалы нельзя считать равноценными акустическому forced alignment.
- Лексическая PER-выборка не является независимой экспертной транскрипцией. Для строгой оценки нужен отдельный ручной phonetic gold set.